# Imports

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from ppmi_utils import (
    cohort_map,
    event_id_to_visit,
    parse_imaging_protocol,
    plot_bar,
    plot_hist,
    primdiag_map,
    visit_to_event_id,
)
from upsetplot import UpSet, from_indicators

CURRENT_DIR = Path(os.getcwd()).resolve()
ROOT_DIR = CURRENT_DIR.parents[1]
DATA_DIR = ROOT_DIR / "data"
IDA_SEARCH_CSV = DATA_DIR / "ppmi" / "ida_search" / "idaSearch_18Feb2026.csv"
CURATED_CSV = DATA_DIR / "ppmi" / "docs" / "PPMI_Curated_Data_Cut_Public_20251112.xlsx"

### idaSearch

In [ ]:
ida_df = pd.read_csv(IDA_SEARCH_CSV, low_memory=False)
print(len(ida_df))
print(ida_df.columns.tolist())
ida_df.head(2)

In [ ]:
# Imaging Protocol parsing
# Automatically get all unique keys
protocol_parsed = ida_df["Imaging Protocol"].apply(parse_imaging_protocol)
all_keys = set()
protocol_parsed.apply(lambda d: all_keys.update(d.keys()))

# Create new columns dynamically
for k in all_keys:
    ida_df[k] = protocol_parsed.apply(lambda x: x.get(k))

print(ida_df.columns.tolist())
ida_df[list(all_keys)].head(2)

In [ ]:
# value count for each column
for k in [
    "Project",
    "Sex",
    "Research Group",
    "Modality",
    "Type",
    "Image Type",
    "Slice Thickness",
    "Acquisition Type",
    "Weighting",
    "Gradient Directions",
    "Field Strength",
    "Acquisition Plane",
    "Matrix Z",
]:
    print(f"Value counts for {k}:")
    print(ida_df[k].value_counts())
    print("\n")

# Advanced modalities

The "Modality" column cannot be trusted, some DTI are in MRI and vice versa, nor the "Weighting" (under Imaging Protocol). So rely on the "Advanced_Modality" column, which is derived from the descriptions, for modality-specific analyses


In [ ]:
# Filter out ignored descriptions
PPMI_IMAGING_IGNORED_PATH = CURRENT_DIR / "ppmi_imaging_ignored.csv"
ignore_df = pd.read_csv(PPMI_IMAGING_IGNORED_PATH)
ignore_set = set(ignore_df["Description"].str.strip().str.upper())
ida_df["Description_clean"] = ida_df["Description"].astype(str).str.strip().str.upper()
ida_df = ida_df[~ida_df["Description_clean"].isin(ignore_set)].copy()

In [ ]:
# avanced modalities, normalize by stripping and uppercasing
PPMI_IMAGING_DESCRIPTIONS_PATH = CURRENT_DIR / "ppmi_imaging_descriptions.json"

with open(PPMI_IMAGING_DESCRIPTIONS_PATH, "r") as f:
    modality_dict = json.load(f)


def normalize_list(lst):
    return [s.strip().upper() for s in lst]


dwi_set = set(normalize_list(modality_dict["dwi"]))
func_set = set(normalize_list(modality_dict["func"]))
anat_sets = {k: set(normalize_list(v)) for k, v in modality_dict["anat"].items()}


In [ ]:
def classify_advanced(row):
    desc = row["Description_clean"]
    if desc in dwi_set:
        return "DWI"
    if desc in func_set:
        return "rsfMRI"
    for anat_type, anat_set in anat_sets.items():
        if desc in anat_set:
            return anat_type  # T1w, T2w, T2starw, FLAIR
    if row["Modality"] == "PET":
        return "PET"
    if row["Modality"] == "SPECT":
        return "SPECT"
    return "Unknown"


ida_df["Advanced_Modality"] = ida_df.apply(classify_advanced, axis=1)

anat_union = set().union(*anat_sets.values())


def classify_bids_modality_fast(row):
    desc = row["Description_clean"]
    if desc in dwi_set:
        return "dwi"
    if desc in func_set:
        return "func"
    if desc in anat_union:
        return "anat"
    return "unknown"


ida_df["BIDS_Modality"] = ida_df.apply(classify_bids_modality_fast, axis=1)
print(ida_df["Advanced_Modality"].value_counts())
# print(pd.crosstab(ida_df["Modality"], ida_df["Advanced_Modality"]))

# Merge with curated clinical data

In [ ]:
clinical_df = pd.read_excel(CURATED_CSV, sheet_name="20251013")

In [ ]:
print(len(clinical_df.columns.to_list()))
print(clinical_df.shape)
print(ida_df.shape)
print(
    "unique patient-study-date pairs ",
    clinical_df.groupby(["PATNO", "visit_date"]).ngroups,
)
print("unique patient-event pairs ", clinical_df.groupby(["PATNO", "EVENT_ID"]).ngroups)
print(clinical_df["enroll_phase"].value_counts())  # June 2020 (Phase 1), then Phase 2

In [ ]:
# add columns for merging

ida_df["Subject ID"] = ida_df["Subject ID"].astype(str)
ida_df["PATNO"] = ida_df["Subject ID"]
ida_df["EVENT_ID"] = ida_df["Visit"].map(visit_to_event_id)
ida_df["Visit"] = ida_df["EVENT_ID"].map(event_id_to_visit)

# prepare clinical dataframe
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)
clinical_df["PRIMDIAG_DESC"] = clinical_df["PRIMDIAG"].map(primdiag_map)
clinical_df["COHORT_DESC"] = clinical_df["COHORT"].map(cohort_map)

# merge clinical and imaging dataframes
# If there are multiple matches for the same (PATNO, EVENT_ID) pair,
# pd.merge() creates a Cartesian product
# it multiplies out all combinations
df = pd.merge(
    clinical_df,
    ida_df,
    # how="left",  # left merge to keep all clinical data, even those without imaging, but results in many NaNs in imaging columns
    # how="right",  # right merge to keep all imaging data, but loses clinical data for subjects without imaging
    how="outer",  # to keep all data, NaN gaps where one side has no match
    on=["PATNO", "EVENT_ID"],
    suffixes=("", "_idasearch"),  # if conflict, Imaging gets '_ida' suffix
)

print("Final merged DataFrame shape:", df.shape)
print("Final merged DataFrame columns:", df.columns.tolist())
print(df["EVENT_ID"].unique().tolist())

# type conversion
cols = ["PATNO", "analytic_subgroup", "Radioisotope"]
df[cols] = df[cols].astype("string")

# YYYYMMDD format for session,
df["Study Date"] = pd.to_datetime(df["Study Date"], errors="coerce")

# create easier references
df["subject"] = df["PATNO"]
df["age_at_visit"] = df["Age"]
df["diagnosis"] = df["Research Group"]
df["session"] = pd.to_datetime(df["Study Date"]).dt.strftime("%Y%m%d")
# drop because not the real age at scan, but age at baseline, which is not relevant
df.drop(columns=["age", "SEX"], inplace=True)

# Filters

In [ ]:
print("Before filter:")
print("Shape ", df.shape)
print("unique patient ", df["subject"].nunique())
print("unique patient-study-date pairs ", df.groupby(["subject", "session"]).ngroups)
print("unique patient-event pairs ", df.groupby(["subject", "Visit"]).ngroups)


In [ ]:
# prepare
df["Weight"] = df["Weight"].replace(0, np.nan)
df["Age"] = df["Age"].replace(0, np.nan)
df["Age"] = df["age_at_visit"].replace(0, np.nan)
df["Study Date"] = pd.to_datetime(df["Study Date"], errors="coerce")

# ida search csv load and preprocessing
df = df[df["Sex"].isin(["M", "F"])].copy()

# add "SWEDD" ?
df = df[df["Research Group"].isin(["PD", "Prodromal", "Control"])].copy()

df = df[(df["Type"] == "Original") | (df["Type"].isna())].copy()

# the protocol 2.0 started in june or august 2020, so we keep only data from then on to have a more homogeneous dataset
# and also because older data is more likely to be of lower quality
df = df[df["Study Date"] >= pd.Timestamp("2020-09-01")].copy()

# filter on descriptions, remove the raws with description that contains one of ['phantom', 'adc', 'trace', localizer']
df = df[
    ~df["Description_clean"].str.contains(
        r"phantom|adc|trace|localizer", case=False, na=False
    )
].copy()

# Most relevant modalities. Neuromanalin is included in T1w  (BIDS convention ?)
df = df[(df["Advanced_Modality"].isin(["DWI", "T1w"]))].copy()

# if DWI, at least 32 gradient directions
df = df[(df["Advanced_Modality"] != "DWI") | (df["Gradient Directions"] >= 16)].copy()

# Keep only rows where a description is used by at least n different patients
# to filter out rare descriptions that less trustworthy
occurence_threshold = 5
df = df[
    df.groupby("Description_clean")["PATNO"].transform("nunique") >= occurence_threshold
].copy()


# ADDITIONAL FILTERS

# RE_NEUROMELANIN = r"([nN][mM])|([gG][rR][eE].*[mM][tT])"
# df = df[
#     df["Description_clean"].str.contains(RE_NEUROMELANIN, case=False, na=False)
# ].copy()

In [ ]:
print("After filter:")
print("Shape ", df.shape)
print("unique patient ", df["subject"].nunique())
print("unique patient-study-date pairs ", df.groupby(["subject", "session"]).ngroups)
print("unique patient-event pairs ", df.groupby(["subject", "Visit"]).ngroups)


# Save CSV

In [ ]:
output_path = CURRENT_DIR / "outputs"
time = datetime.now().strftime("%Y%m%d_%H%M%S")
# version without duplicates for metadata use independant of the scan itself (clinical data, demographics, etc.)
df_no_duplicates = df.drop_duplicates(subset=["subject", "session"], keep="first")
df_no_duplicates.to_csv(output_path / f"ppmi_merged_{time}.csv", index=False)
# for completeness, save the full dataframe with duplicates as well, containing  the used IDs
df.to_csv(output_path / f"ppmi_merged_full_{time}.csv", index=False)

print(f"DataFrame saved to {output_path}")

In [ ]:
len(df_no_duplicates)
len(df)

In [ ]:
# print most recent study date
print("Most recent study date:", df["Study Date"].max())

# Subject and Image IDs
Used to download the data

In [ ]:
n = 10  # number of chunks

image_ids = (
    df["Image ID"]
    .dropna()
    .apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
    .unique()
    .tolist()
)

print(f"Total unique Image IDs: {len(image_ids)}")

# Compute chunk size (ceiling division)
chunk_size = (len(image_ids) + n - 1) // n

# save in txt file
output_txt_path = (
    CURRENT_DIR
    / "outputs"
    / f"ppmi_image_ids_chunks_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
)
with open(output_txt_path, "w") as f:
    for i in range(n):
        chunk = image_ids[i * chunk_size : (i + 1) * chunk_size]
        if not chunk:
            continue  # skip empty chunks if n > len(image_ids)
        f.write(f"Chunk {i + 1}:\n")
        f.write(f"Length: {len(chunk)}\n")
        f.write(",".join(chunk) + "\n\n")
        print(f"\nChunk {i + 1}:")
        print(f"Length: {len(chunk)}")
        print(",".join(chunk))

# Descriptions

In [ ]:
description_clean_list = df["Description_clean"].unique().tolist()
print("Number of unique cleaned descriptions:", len(description_clean_list))
print(description_clean_list)
print(df["Advanced_Modality"].unique().tolist())

# Get the list
desc_list = df["Description_clean"].unique().tolist()

# Save to a txt file
with open("outputs/unique_descriptions.txt", "w", encoding="utf-8") as f:
    f.writelines(str(item) + "\n" for item in desc_list)

In [ ]:
desc_counts = df["Description_clean"].value_counts()
top_df = desc_counts.reset_index()
top_df.columns = ["Description", "Count"]

# Reverse order for horizontal readability (small → big from top to bottom)
top_df = top_df.iloc[::-1]

fig = px.bar(
    top_df,
    x="Count",
    y="Description",
    orientation="h",
    title="Descriptions",
    height=2000,
)

# Remove categoryorder — let your dataframe control it
fig.update_layout(
    yaxis={"categoryorder": "array", "categoryarray": top_df["Description"]}
)
fig.update_yaxes(
    tickmode="array",
    tickvals=top_df["Description"],
    ticktext=top_df["Description"],
    tickfont=dict(size=8),  # shrink font so all labels fit
)
fig.show()

In [ ]:
# print percetage of na values in each column in the following list
col_list = ["PRIMDIAG", "COHORT", "Research Group", "subgroup", "Image ID"]
for col in col_list:
    na_percentage = df[col].isna().mean() * 100
    print(f"{col}: {na_percentage:.2f}% NA values")

# Basic analysis

### Age at earliest visit

In [ ]:
df["age_at_earliest_visit"] = df.groupby("PATNO")["age_at_visit"].transform("min")
print(df[["age_at_earliest_visit"]].describe().T)
plot_hist(df, "age_at_earliest_visit", title="Age at earliest visit")

### Cohort definition
1. Parkinson’s Disease, i.e., people who have a formal diagnosis of Parkinson’s disease (PD) 
2. Prodromal, i.e., people who are at risk of developing PD based on clinical features, genetic variants or other biomarkers but have not been formally diagnosed 
3. Healthy Controls, i.e., people with no neurologic disorder and no first-degree relative with PD
4. SWEDD (Scan without dopaminergic deficit). This is a small legacy cohort that you may wish to exclude, depending on your research purpose; for more details, see https://www.ppmi-info.org/study-design/study-cohorts#legacy/

In [ ]:
# diagnosis at baseline
earliest_visit_df = df[df["age_at_visit"] == df["age_at_earliest_visit"]].copy()
baseline_unique = earliest_visit_df.drop_duplicates(
    subset=["PATNO"]
)  # keep first occurence
cohort_counts = baseline_unique["Research Group"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "Research Group",
    title="Research Group distribution at earliest visit",
)

In [ ]:
cohort_counts = baseline_unique["COHORT_DESC"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "COHORT_DESC",
    title="Cohort distribution at earliest visit",
)

### Subgroup
Subgroup is derived from various source columns to give a more detailed group assignment than cohort. It can take values of Healthy Control, SWEDD, SWEDD/PD, SWEDD/nonPD, Hyposmia, RBD, Sporadic PD, LRRK2, GBA, PINK1, PRKN, SNCA or combinations of genetic variants and/or RBD (e.g. LRRK2 + GBA, GBA + RBD).

In [ ]:
cohort_counts = baseline_unique["subgroup"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "subgroup",
    title="Subgroups distribution at earliest visit",
)

### PRIMDIAG

In [ ]:
cohort_counts = baseline_unique["PRIMDIAG_DESC"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "PRIMDIAG_DESC",
    title="PRIMDIAG distribution at earliest visit",
)

### Cross tab for diag, subgroup, research group

In [ ]:
print(pd.crosstab(baseline_unique["PRIMDIAG_DESC"], baseline_unique["subgroup"]))


# Create crosstab
ct = pd.crosstab(
    baseline_unique["PRIMDIAG_DESC"],
    baseline_unique["Research Group"],
    # baseline_unique["subgroup"],
)

# Optional: sort by total frequency
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]

# Plot heatmap
plt.figure(figsize=(10, 12))

sns.heatmap(
    ct,
    annot=True,  # show raw numbers
    fmt="d",  # integer formatting
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "Count"},
)

plt.title("Primary Diagnosis by Research Group (Raw Counts)")
plt.xlabel("Research Group")
plt.ylabel("Primary Diagnosis")
plt.tight_layout()
plt.show()

# Longitudinal analysis

### Cumulative number of visits through years

In [ ]:
# Keep only unique patient-event pairs
unique_visits = df.drop_duplicates(subset=["PATNO", "EVENT_ID"])
unique_visits = unique_visits[unique_visits["Study Date"] >= "2010-01-01"]

# Extract Year-Month
unique_visits["YearMonth"] = unique_visits["Study Date"].dt.to_period("M")

# Count visits per month
visits_per_month = unique_visits.groupby("YearMonth").size()
visits_per_month = visits_per_month.cumsum().reset_index()
visits_per_month.columns = ["YearMonth", "CumulativeVisits"]

# Convert YearMonth to timestamp for Plotly
visits_per_month["YearMonth"] = visits_per_month["YearMonth"].dt.to_timestamp()

# Plot interactive line chart
fig = px.line(
    visits_per_month,
    x="YearMonth",
    y="CumulativeVisits",
    title="Cumulative Number of Unique Visits per Month",
    markers=True,
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Unique Visits",
    xaxis=dict(rangeslider=dict(visible=True)),  # adds a zoomable range slider
    hovermode="x unified",
)

fig.show()

### How many visits per patient ?

In [ ]:
visits_per_patient = df.groupby("PATNO")["EVENT_ID"].nunique()
print("Visits per patient (summary):")
print(visits_per_patient.describe().to_frame().T)
# Histogram of visit counts
plot_hist(
    visits_per_patient.reset_index(), "EVENT_ID", title="Number of visits per patient"
)

### UpSet Plot

In [ ]:
# Keep unique patient-event pairs
# df_unique = df[["PATNO", "EVENT_ID"]].drop_duplicates()
df_unique = df[["PATNO", "Visit"]].drop_duplicates()

# Create binary matrix: rows = patients, columns = EVENT_ID
df_binary = (
    df_unique.assign(value=1)
    # .pivot_table(index="PATNO", columns="EVENT_ID", values="value", fill_value=0)
    .pivot_table(index="PATNO", columns="Visit", values="value", fill_value=0)
    .astype(bool)
)

# Convert to UpSet format
upset_data = from_indicators(df_binary.columns.tolist(), df_binary)

# Plot
upset = UpSet(upset_data, subset_size="count", show_counts=True, sort_by="cardinality")

upset.plot()
plt.title("Patient overlap across EVENT_IDs")
plt.show()

### Longitudinal diagnosis changes

There is no change for 
column_name_for_transition = "COHORT_DESC"
column_name_for_transition = "Research Group"

In [ ]:
column_name_for_transition = "PRIMDIAG_DESC"

# number of distinct diagnoses per patient
diagnosis_over_time = df.groupby("PATNO")[column_name_for_transition].nunique()

# Patients with >1 diagnosis
patients_changed_diag = diagnosis_over_time[diagnosis_over_time > 1]
print(f"Number of patients with diagnosis change: {len(patients_changed_diag)}")

# diagnosis_per_patient = df.groupby("PATNO")[column_name].unique()
# for pat in patients_changed_diag.index:
#     print(f"{pat}: {diagnosis_per_patient[pat]}")

# Plot distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=diagnosis_over_time, color="skyblue")
plt.title("Number of distinct diagnoses per patient")
plt.xlabel("Distinct diagnoses count")
plt.ylabel("Number of patients")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# Ensure visits are ordered in time
df_sorted = df.sort_values(["PATNO", "Study Date"])

transitions = []

for pat, group in df_sorted.groupby("PATNO"):
    diagnoses = group[column_name_for_transition].dropna().unique()

    if len(diagnoses) > 1:
        # Get ordered diagnoses per visit (not just unique)
        ordered_diag = (
            group[[column_name_for_transition, "Study Date"]]
            .dropna()
            .sort_values("Study Date")[column_name_for_transition]
            .tolist()
        )

        # Remove consecutive duplicates
        cleaned = [ordered_diag[0]]
        for d in ordered_diag[1:]:
            if d != cleaned[-1]:
                cleaned.append(d)

        # Create pairwise transitions
        for i in range(len(cleaned) - 1):
            transitions.append(f"{cleaned[i]} → {cleaned[i + 1]}")


transition_counts = pd.Series(transitions).value_counts()
print(len(transitions))
transition_counts = transition_counts.head(5)  # keep top 10 transitions

# Total number of transitions
total_transitions = transition_counts.sum()

plt.figure(figsize=(10, 5))

ax = sns.barplot(
    x=transition_counts.values, y=transition_counts.index, color="steelblue"
)

plt.title("Diagnosis Transitions (Patients with Change)")
plt.xlabel("Number of Patients")
plt.ylabel("Transition")

# Annotate with count + percentage
for i, (count, label) in enumerate(
    zip(transition_counts.values, transition_counts.index)
):
    percentage = 100 * count / total_transitions
    ax.text(
        count + 0.5,  # position slightly to the right of bar
        i,
        f"{count} ({percentage:.1f}%)",
        va="center",
    )

plt.tight_layout()
plt.show()


# print("Transition counts:")
# print(transition_counts)

### Retention

In [ ]:
# Ensure datetime
visite_date_column = "visit_date"
df[visite_date_column] = pd.to_datetime(df[visite_date_column], errors="coerce")

# Keep unique patient-event pairs
df_unique = df.drop_duplicates(subset=["PATNO", "EVENT_ID"]).copy()

# Baseline date per patient
baseline_dates = (
    df_unique[df_unique["EVENT_ID"] == "BL"].groupby("PATNO")[visite_date_column].min()
)
df_unique["baseline_date"] = df_unique["PATNO"].map(baseline_dates)

# Time from baseline in months
df_unique["months_from_baseline"] = (
    df_unique["visit_date"] - df_unique["baseline_date"]
).dt.days / 30.44
df_unique = df_unique[df_unique["months_from_baseline"] >= 0].dropna(
    subset=["months_from_baseline"]
)

# Plot as a curve
df_plot = df_unique[df_unique["months_from_baseline"] <= 60].copy()
bins = np.arange(0, df_plot["months_from_baseline"].max() + 1, 3)
counts, edges = np.histogram(df_plot["months_from_baseline"], bins=bins)

# Midpoints of bins for plotting
bin_centers = (edges[:-1] + edges[1:]) / 2

plt.figure(figsize=(10, 6))
plt.plot(bin_centers, counts, marker="o", linestyle="-", color="steelblue")
plt.title(
    "Follow-up Visits Over Time (Months from Baseline) - bins of 3 months - up to 60 months"
)
plt.xlabel("Months from Baseline")
plt.ylabel("Number of Visits")
plt.grid(axis="y", alpha=0.3)
plt.xticks(np.arange(0, 60 + 1, 3))

plt.tight_layout()
plt.show()


In [ ]:
# Ensure datetime
df_unique = df.drop_duplicates(subset=["PATNO", "EVENT_ID"]).copy()
df_unique["visit_date"] = pd.to_datetime(df_unique["visit_date"], errors="coerce")

# Get baseline date per patient
baseline_dates = (
    df_unique[df_unique["EVENT_ID"] == "BL"].groupby("PATNO")["visit_date"].min()
)
df_unique["baseline_date"] = df_unique["PATNO"].map(baseline_dates)

# Compute months from baseline
df_unique["months_from_baseline"] = (
    df_unique["visit_date"] - df_unique["baseline_date"]
).dt.days / 30.44
df_unique = df_unique[df_unique["months_from_baseline"] >= 0]

# Compute total visits per patient
visits_per_patient = df_unique.groupby("PATNO")["EVENT_ID"].nunique()


# Assign subgroups based on total visits
def assign_visit_group(n):
    if n == 1:
        return "1 visit"
    elif n == 2:
        return "2 visits"
    else:
        return "3+ visits"
    # elif n == 3:
    #     return "3 visits"
    # elif n == 4:
    #     return "4 visits"

    # else:
    #     return "5+ visits"


df_unique["visit_group"] = (
    df_unique["PATNO"].map(visits_per_patient).map(assign_visit_group)
)
df_plot = df_unique[df_unique["months_from_baseline"] <= 60].copy()


# Plot
plt.figure(figsize=(10, 6))
bins = np.arange(0, df_plot["months_from_baseline"].max() + 1, 2)
for group_name, group_df in df_unique.groupby("visit_group"):
    counts, bin_edges = np.histogram(group_df["months_from_baseline"], bins=bins)
    # Convert counts to cumulative or normalized fraction if desired
    plt.plot(bin_edges[:-1], counts, marker="o", label=group_name)


# Plot full population
counts_all, _ = np.histogram(df_unique["months_from_baseline"], bins=bins)
plt.plot(
    bin_edges[:-1],
    counts_all,
    marker="o",
    color="black",
    linestyle="--",
    label="All patients",
)


plt.title("Follow-up Visits over Time by Patient Subgroup")
plt.xlabel("Months from Baseline")
plt.ylabel("Number of Visits")
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Patient Subgroup")
plt.xticks(np.arange(0, 60 + 1, 3))

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
bins = np.arange(0, df_plot["months_from_baseline"].max() + 1, 2)
# group_name = "visit_group"
group_name = "Research Group"
# Plot one curve per subgroup
for gp, group_df in df_plot.groupby(group_name):
    # Get subgroup size (number of unique patients)
    n_patients = group_df["PATNO"].nunique()

    # Histogram counts
    counts, bin_edges = np.histogram(group_df["months_from_baseline"], bins=bins)

    # Convert to percentage relative to subgroup
    perc = counts / n_patients * 100

    plt.plot(bin_edges[:-1], perc, marker="o", label=gp)

# Plot full population
total_patients = df_unique["PATNO"].nunique()
counts_all, _ = np.histogram(df_unique["months_from_baseline"], bins=bins)
perc_all = counts_all / total_patients * 100
plt.plot(
    bin_edges[:-1],
    perc_all,
    marker="o",
    color="black",
    linestyle="--",
    label="All patients",
)


plt.title("Follow-up Visits over Time by Patient Subgroup (≤ 60 months)")
plt.xlabel("Months from Baseline")
plt.ylabel("Percentage of Subgroup (%)")
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Patient Subgroup")

# X-axis ticks every 3 months
plt.xticks(np.arange(0, 60 + 1, 3))

plt.tight_layout()
plt.show()


- Tous on une baseline
- si 2 visits, la deuxieme est surtout à +12 et parfois +24 ou +48
- si 3, 2 suivant surtout sur +12 +24
- si 4, les trois apres BL sont equitablement réparties sur +12 +24 +48
- si 5+, bonne répartition avec du +36

# Image analysis

### Basic info on rows level

In [ ]:
df_image = df[df.columns.intersection(ida_df.columns)]
print("Image-related DataFrame shape:", df_image.shape)
print("Image-related DataFrame columns:", df_image.columns.tolist())

### histograms

In [ ]:
# Separate numeric and categorical columns
# numeric_cols = df_image.select_dtypes(include=np.number).columns
numeric_cols = ["Slice Thickness", "Matrix Z", "Field Strength"]
# numeric_cols = []

# -------------------------------
# Numeric Columns Histograms
# -------------------------------
for col in numeric_cols:
    plt.figure(figsize=(6, 4))

    # Plot histogram
    ax = sns.histplot(
        df_image[col].dropna(), bins=30, kde=False
    )  # disable KDE for counts clarity

    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")

    # Annotate counts on top of each bin
    for patch in ax.patches:
        height = patch.get_height()
        if height > 0:  # only annotate non-empty bins
            ax.text(
                patch.get_x() + patch.get_width() / 2,  # center of bin
                height + 0.5,  # slightly above the bar
                int(height),  # show integer count
                ha="center",
                va="bottom",
                fontsize=8,
            )

    plt.show()

# -------------------------------
# Categorical Columns Bar Plots
# -------------------------------
categorical_cols = [
    # "Visit",
    # "Sex",
    # "Research Group",
    "Modality",
    "Advanced_Modality",
    "Type",
    # "Structure",
    # "Laterality",
    # "Image Type",
    # "Registration",
    "Description",
    # "Tissue",
    # "Acquisition Plane",
    "Acquisition Type",
    # "Manufacturer",
    # "Mfg Model",
    # "Weighting",
]

categorical_cols = [col for col in categorical_cols if col in df_image.columns]

for col in categorical_cols:
    plt.figure(figsize=(6, 4))

    df_image[col] = df_image[col].fillna("Unknown").astype(str)

    # Compute counts after cleaning
    counts = df_image[col].value_counts()
    total = counts.sum()
    order = counts.index.tolist()  # already strings

    ax = sns.countplot(y=col, data=df_image, order=order)

    for p, category in zip(ax.patches, order):
        count = counts.get(category, 0)
        percent = 100 * count / total

        ax.text(
            p.get_width() + 0.5,
            p.get_y() + p.get_height() / 2,
            f"{count} ({percent:.1f}%)",
            va="center",
            fontsize=8,
        )

    plt.title(f"Value Counts for {col}")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


### Image descriptions

In [ ]:
# COunt the values of "Image Description"
print(df_image["Description"].value_counts(dropna=False).head(10))

### Simultaneous Multimodal Availability (Per Visit)

In [ ]:
# Unique subject-visit-modality combinations
visit_modalities = (
    df.groupby(["PATNO", "EVENT_ID"])["Advanced_Modality"].unique().reset_index()
)

visit_modalities["n_modalities"] = visit_modalities["Advanced_Modality"].apply(len)
print(visit_modalities["n_modalities"].value_counts().to_frame().T)
print(visit_modalities["n_modalities"].describe().to_frame().T)


In [ ]:
visit_modalities.head(2)

In [ ]:
# Unique subject-visit-modality combinations
visit_modalities = (
    df.groupby(["PATNO", "EVENT_ID"])["Advanced_Modality"].unique().reset_index()
)

# Explode so each row is one modality per visit
exploded = visit_modalities.explode("Advanced_Modality")

# Count visits per modality
modality_counts = (
    exploded["Advanced_Modality"].value_counts().sort_values(ascending=False)
)

# Plot
plt.figure(figsize=(12, 6))
plt.bar(modality_counts.index, modality_counts.values)

plt.title("Number of Visits per Modality")
plt.xlabel("Advanced Modality")
plt.ylabel("Number of Visits")
plt.xticks(rotation=45, ha="right")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create modality combination string
visit_modalities["modality_combo"] = visit_modalities["Advanced_Modality"].apply(
    lambda x: " | ".join(sorted(x))
)

# Count number of visits per combination
visit_counts = visit_modalities["modality_combo"].value_counts()

# Count number of unique patients per combination
patient_counts = (
    visit_modalities.groupby("modality_combo")["PATNO"]
    .nunique()
    .sort_values(ascending=False)
)

# Combine both in one table
summary = visit_counts.to_frame("n_visits").join(patient_counts.to_frame("n_patients"))

# print(summary)

In [ ]:
# Optional: keep only top 15 combinations for readability
top_n = 8
summary_plot = summary.sort_values("n_visits", ascending=False).head(top_n)

plt.figure(figsize=(10, 8))

ax = sns.barplot(x=summary_plot["n_visits"], y=summary_plot.index, color="steelblue")

plt.title("Top Modality Combinations per Visit")
plt.xlabel("Number of Visits")
plt.ylabel("Modality Combination")

# Annotate counts
for i, (visits, patients) in enumerate(
    zip(summary_plot["n_visits"], summary_plot["n_patients"])
):
    ax.text(visits + 1, i, f"{visits} visits | {patients} patients", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Pivot to binary presence matrix
visit_modality_matrix = (
    df.assign(present=1)
    .drop_duplicates(["PATNO", "EVENT_ID", "Advanced_Modality"])
    .pivot_table(
        index=["PATNO", "EVENT_ID"],
        columns="Advanced_Modality",
        values="present",
        fill_value=0,
    )
)
co_occurrence = visit_modality_matrix.T @ visit_modality_matrix
print(co_occurrence)

In [ ]:
df.head(2)